<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP7.2A/MP7.2A_moving_hot_spot.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP7.2A · A Die with a Moving Hot Spot

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/")
PROJECT = (['mp72a.py', 'MP7.2A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP7.2A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex07.2-heat-and-wave", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex07.2-heat-and-wave", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp72a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp72a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP7.2A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex07.2's problem.py, by its name in Ex07.2

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

A processor or a power chip is not heated evenly. Its power is made in a few **blocks** - the cores, a cache, an arithmetic unit, a switching cell - and the work moves between them: a scheduler hands a task from one core to another, a converter commutates from one switch to the next. Each block that switches on starts a local hot spot, and the hot spot **moves** with the work. The temperature that matters is the **peak**: the highest temperature anywhere in the die, and when it is reached. Above about 100 °C a chip throttles its clock, ages faster and, in the end, fails.

So a thermal designer, or the firmware that decides how to spread the work, wants the field of the die and its peak **for every power the blocks may run at** - many times, for every schedule considered. Ex07.2's first notebook let a die cool down from a given field. This mini project powers it: three blocks switching on in turn, and silicon whose conductivity falls as it warms. It builds a network that answers a **new power** in a fraction of the time a conventional solver needs.

In [ ]:
# The die, its blocks and its data
fig, ax = plt.subplots(figsize=(6.4, 4.8))
mp.draw_die(ax)                                            # the die, the blocks and when each is on
plt.show()
rows = [("die", f"{mp.L*1e3:.0f} × {mp.L*1e3:.0f} mm silicon, {mp.THICK*1e3:.1f} mm thick; edges held at the package's {mp.T_PKG:.0f} °C"),
        ("blocks", f"three, {mp.BLOCK*1e3:.0f} × {mp.BLOCK*1e3:.0f} mm, centred at (2.5, 2.5), (7.5, 2.5) and (5.0, 7.5) mm"),
        ("schedule", "block 1 on from 0 to 60 ms, block 2 from 60 to 120 ms, block 3 from 120 to 180 ms; all off to 200 ms"),
        ("power", f"P = {mp.P_MIN:.0f} to {mp.P_MAX:.0f} W per block, the same for the three; {mp.P0:.0f} W is {mp.q_block(mp.P0)*mp.THICK/1e4:.0f} W/cm²"),
        ("heat density", f"q = P / (block area × thickness) = {mp.q_block(mp.P0)/1e9:.1f} GW/m³ at {mp.P0:.0f} W"),
        ("silicon", f"k = {mp.K300:.0f} (300 K / T)^{mp.K_EXP} W/(m·K): {mp.k_si(0.0):.1f} at 65 °C, {mp.k_si(35.0):.1f} at 100 °C"),
        ("heat capacity", f"ρc = {mp.RHO_C/1e6:.2f} MJ/(m³K), so that k/ρc at 300 K is Ex07.2's {pb.ALPHA:.1e} m²/s"),
        ("window", f"0 to {mp.T_END*1e3:.0f} ms, starting at the package temperature everywhere")]
for k, v in rows:
    print(f"{k:>14}  {v}")

**What you should see.** The die of Ex07.2 seen from above, its edges held by the package at 65 °C, and three blocks: two near the lower corners, one near the top edge. Each is on for 60 ms, in turn, and the die then cools for 20 ms. At 8 W a block puts 200 W/cm² through the die, a hot spot's flux in a modern processor. Silicon's conductivity is 126.7 W/(m·K) at 65 °C and 111.4 at 100 °C: 12 % less where the die is hottest. **The die's thickness, the block power and the fit for $k(T)$ are assumed** - typical values; the fit is the usual one to the measured conductivity of pure silicon (Glassbrenner and Slack, 1964), written from memory and to be checked.

## A2 · What is new against Ex07.2

| | Ex07.2, notebook 1 | this mini project |
|---|---|---|
| the start | a given field, 60 K above the package at its peak | the package temperature everywhere |
| the heat | none: the die only cools | three blocks of $P$ watts, each on for 60 ms in turn |
| the conductivity | one constant, 148 W/(m·K) | silicon's $k(T)$: it falls as the die warms |
| the equation | linear | **nonlinear**: $k$ depends on the answer |
| the solution | exact: two decaying modes | **none exists**: numerical only |
| what is asked | the field of one case | the field and its peak for **any power** from 5 to 10 W |

**Why no exact solution exists.** Ex07.2's exact solution works because the equation is linear with one constant conductivity: every mode $\sin(m\pi x/L)\sin(n\pi y/L)$ of the square decays by itself, and any field is a sum of them. With $k(T)$ the equation is nonlinear - the conductivity, and with it the way heat spreads, depends on the temperature being computed - and the modes no longer separate. That is the normal case in engineering.

**The strength of the change.** The conductivity is written $k = 148\,(300\ \mathrm{K}/T)^{1.3\,S}$. At $S = 1$ it is silicon's; at $S = 0$ it is the constant 148 W/(m·K) of Ex07.2's diffusivity, and with Ex07.2's starting field and no power the solver must reproduce Ex07.2's two exact modes. Part C checks it.

## A3 · The equation and its conditions

The unknown is the temperature rise above the package, $\theta(x, y, t) = T - 65$ °C, which depends also on the power $P$ of the blocks. Heat stored in a small area changes its temperature, heat conducted in or out and heat made there change what is stored:

$$\rho c\,\frac{\partial \theta}{\partial t} = \nabla\cdot\big(k(\theta)\,\nabla\theta\big) + q(x, y, t; P) \quad\text{in the die}, \qquad q = \begin{cases} P/(A_\mathrm{block}\,t_\mathrm{die}) & \text{in the block that is on}\\ 0 & \text{elsewhere}\end{cases}$$

| where | condition | why it holds |
|---|---|---|
| the four edges | $\theta = 0$ | the package and its lid hold the die's edges at 65 °C; Ex07.2's condition |
| the start, $t = 0$ | $\theta = 0$ | the die has been idle at the package temperature |
| the faces of the die | no heat crosses them | a two-dimensional model, as in Ex07.2: the heat leaves through the edges (assumed; a real die also loses heat through its back face) |
| the edge of a block | $\theta$ and $k\,\partial\theta/\partial n$ continuous | the block is the same silicon; only its heat stops at its edge |

Two features of $q$ matter for a network. **In space** it jumps at the edges of the block, so the curvature of $\theta$ jumps there. **In time** it jumps at every switch (0, 60, 120 and 180 ms), so the rate of change of $\theta$ jumps inside the blocks. The field itself stays smooth: diffusion rounds both jumps, within about $\sqrt{\alpha t}$ = 0.3 mm one millisecond after a switch.

In [ ]:
# The ground truth at 8 W: the field at three instants, and the peak against time
G = np.load("MP7.2A_truth.npz")                            # the ground truth (part C explains how it is made)
x = G["x"] * 1e3
F = G["field_8W"]                                         # (101 instants, 41, 41) [K]
lin = mp.Linear(60)                                       # the same die with k held at its 65 °C value: exact
F_lin = lin.grid(G["x"], G["t"])                          # (101, 41, 41) [K]
fig, ax = plt.subplots(1, 4, figsize=(16, 3.7), gridspec_kw={"width_ratios": [1, 1, 1, 1.5]})
for a, tt in zip(ax[:3], (0.058, 0.118, 0.18)):
    i = int(round(tt / 0.002))
    im = a.contourf(x, x, F[i], levels=np.linspace(0, 34, 18), cmap="inferno")
    a.set_aspect("equal"); a.set_title(f"t = {G['t'][i]*1e3:.0f} ms, max {F[i].max():.2f} K", fontsize=10); a.set_xlabel("x [mm]")
ax[0].set_ylabel("y [mm]"); plt.colorbar(im, ax=ax[2], shrink=0.8, label="θ [K]")
ax[3].plot(G["t"] * 1e3, F.max(axis=(1, 2)), label="silicon's k(T): ground truth")
ax[3].plot(G["t"] * 1e3, F_lin.max(axis=(1, 2)), "--", label="k held at its 65 °C value")
for s in mp.SWITCHES:
    ax[3].axvline(s * 1e3, color="grey", lw=0.5)
ax[3].set_xlabel("t [ms]"); ax[3].set_ylabel("largest rise in the die [K]"); ax[3].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"peak at 8 W: {float(G['peak_8W']):.2f} K above the package ({mp.T_PKG + float(G['peak_8W']):.2f} °C) at "
      f"{float(G['t_peak_8W'])*1e3:.0f} ms; with k held at 65 °C: {float(G['linear_peak_8W']):.2f} K")
print(f"k(T) adds {float(G['peak_8W']) - float(G['linear_peak_8W']):.2f} K to the peak")

**What you should see.** Three snapshots at 8 W, each at the end of a block's 60 ms: the hot spot sits on the block that is on and has moved with the work; the heat of the earlier blocks is still draining to the edges. Right, the largest rise in the die against time: it climbs during each pulse, falls by about half in the 10 ms after each switch - the block just switched off cools faster than the next one heats - and reaches **33.84 K (98.84 °C) at 180 ms**, the end of the third pulse - block 3 has only one edge near it and starts from a die already warmed by the first two. With the conductivity held at its 65 °C value (the dashed line, computed exactly by the die's modes) the peak is 32.17 K: **silicon's $k(T)$ adds 1.67 K**, about 5 %, which a linear model misses.

## A4 · What is asked

| quantity | at | required |
|---|---|---|
| the peak, the largest rise anywhere in the die over the window | the test powers 5.5, 7.25 and 9.5 W, which you do not train on | within **1 K** of the ground truth |
| the instant of the peak | the same powers | within **2 ms** |
| the field $\theta(x, y, t)$ at the 41 × 41 scoring nodes (every 0.25 mm), every 2 ms | the same powers | within **2 K** everywhere, at every instant |
| the time to answer a new power online: the field at the scoring nodes at the 101 instants, and the peak | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C) |

One network, trained once, offline, takes the power as an input: a new power is answered by evaluating it, not by training again. The training time is reported apart, as the offline cost.

**The agreed accuracy.** 1 K on a peak of about 30 K is 3 %, less than what $k(T)$ adds; 2 K on the field is the accuracy the reference model's grid was chosen to meet (part C): its 0.5 mm cells meet both, a coarser grid does not. A die's temperature sensor is rarely better than 1 to 2 K.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

$$(x,\ y,\ t,\ P)\ \longmapsto\ \theta(x, y, t; P)$$

the rise above the package at a point of the die, at an instant, for a power of the blocks. The power is an **input**, like the coordinates and the time: a new power is answered by evaluating the network.

The scales of the problem:

In [ ]:
# The scales of the problem
alpha = mp.k_si(0.0) / mp.RHO_C                           # m²/s, the diffusivity at 65 °C
print(f"diffusivity at 65 °C: {alpha:.2e} m²/s (Ex07.2's {pb.ALPHA:.1e} is at 300 K)")
print(f"time for heat to cross the die, L²/α: {mp.L**2/alpha*1e3:.0f} ms; to cross a block, (2 mm)²/α: {(2e-3)**2/alpha*1e3:.0f} ms")
print(f"reach of diffusion 1 ms after a switch, √(α t): {np.sqrt(alpha*1e-3)*1e3:.2f} mm")
print(f"heating rate of a block at 8 W, before any heat leaves it, q/ρc: {mp.q_block(8.0)/mp.RHO_C:.0f} K/s")
for P in (mp.P_MIN, mp.P0, mp.P_MAX):
    print(f"P = {P:4.1f} W: heat density {mp.q_block(P)/1e9:.1f} GW/m³")

**What you should see.** The diffusivity at 65 °C is 7.53e-05 m²/s. Heat crosses the 10 mm die in about 1 300 ms and a 2 mm block in about 53 ms: a block's 60 ms pulse is long enough for its own hot spot to settle, much too short for the die to settle. One millisecond after a switch diffusion has reached only 0.27 mm: the field near a block that has just switched on is sharp. A block at 8 W heats at 2 378 K/s before any heat leaves it.

## B2 · The physics the network is trained with

In units of order one - $X = x/L$, $Y = y/L$, $\hat t = t/t_\mathrm{end}$ - and with the residual divided by the heat density $q_\mathrm{block}(P)$ so that it is of order one:

$$r = \frac{1}{q_\mathrm{block}(P)}\left[\rho c\,\frac{\partial\theta}{\partial t} - \nabla\cdot\big(k(\theta)\,\nabla\theta\big) - q\right]$$

| loss term | where it is evaluated | what it enforces |
|---|---|---|
| $\mathcal{L}_\mathrm{pde}$ | points inside the die, over the window and the power range | the heat equation with silicon's $k(T)$ |
| $\mathcal{L}_\mathrm{edges}$ | points on the four edges | $\theta = 0$ |
| $\mathcal{L}_\mathrm{start}$ | points at $t = 0$ | $\theta = 0$ |

**What can be built in rather than trained.** The edges hold for any weights if the output is multiplied by $X(1-X)\,Y(1-Y)$, which vanishes on them (Ex07.2's hard enforcement); the start holds if the output is multiplied by a function of $t$ that is zero at $t = 0$.

**The linear part, exactly.** With the conductivity held at its 65 °C value the equation is linear, and Ex07.2's method solves it exactly: a sum of the die's modes $\sin(m\pi x/L)\sin(n\pi y/L)$, each block's heat projected on them, each mode rising and falling as a first-order system with its own time constant. This field, $\theta_\mathrm{lin}$, is proportional to $P$, carries every jump of the heat in space and in time, and is already within 2 K of the answer (section A3). `mp.Linear(M)` computes it with $M \times M$ modes at any points, with its slopes and its Laplacian. A network may then learn only what $k(T)$ adds, $\delta = \theta - \theta_\mathrm{lin}$, which obeys

$$\rho c\,\frac{\partial\delta}{\partial t} = k\,\nabla^2\delta + \big(k - k_{65}\big)\nabla^2\theta_\mathrm{lin} + \frac{dk}{d\theta}\,\big|\nabla\theta_\mathrm{lin} + \nabla\delta\big|^2, \qquad k = k(\theta_\mathrm{lin} + \delta)$$

with $\delta = 0$ on the edges and at the start. Its source is about a tenth of $q$, and it is continuous in time: $\nabla^2\theta_\mathrm{lin}$ does not jump at a switch, because the jump of $\partial\theta/\partial t$ and the jump of $q$ cancel. Whether you use this is your choice in part D.

**If you do not.** A plain network is smooth; the heat jumps at the blocks' edges and at the switches. The lesson of MP8.2A applies: a network that must follow a switching source needs inputs that carry the switching (the time since each switch, or functions of it), and many points just after each switch - but not so many exactly on a jump that the residual it cannot remove dominates the loss.

## B3 · The training points

| function | returns | used for |
|---|---|---|
| `mp.sample_interior(n, seed, near, near_t)` | x, y [m], t [s], P [W]: a share `near` of the points within 2.5 mm of the block that is on, a share `near_t` of the instants just after a switch | $\mathcal{L}_\mathrm{pde}$ |
| `mp.sample_times(n, rng, near)` | instants [s], a share just after a switch | any term in time |
| `mp.heat(x, y, t, P)` | the heat density $q$ [W/m³] at the points | the residual |
| `mp.Linear(M).at(x, y, t)` | $\theta_\mathrm{lin}$ at $P_0$ = 8 W and its two slopes and Laplacian at the points, (4, n) | building in the linear part |
| `mp.Linear(M).grid(xs, ts)` | $\theta_\mathrm{lin}$ at $P_0$ on a grid of nodes at a set of instants | the same, on a grid |

$\theta_\mathrm{lin}$ is proportional to the power: multiply by $P/P_0$. **The test powers, 5.5, 7.25 and 9.5 W**, lie inside the range but are not chosen by any sampler: the network is scored where it was not fitted.

In [ ]:
# The training points, and how far the linear part is from the truth
x, y, t, P = mp.sample_interior(4000, seed=1)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1.2, 1.2]})
for i, c in enumerate(("#d62728", "#ff7f0e", "#9467bd")):
    on, off = mp.BLOCKS[i][1], mp.BLOCKS[i][2] if i < 2 else mp.T_END + 1
    sel = (t >= on) & (t < off)
    ax[0].scatter(x[sel] * 1e3, y[sel] * 1e3, s=1, c=c, label=f"while block {i + 1} is on" + (" or after" if i == 2 else ""))
ax[0].set_aspect("equal"); ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("y [mm]"); ax[0].legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.15))
ax[1].hist(t * 1e3, bins=100); [ax[1].axvline(s * 1e3, color="r", lw=0.6) for s in mp.SWITCHES]
ax[1].set_xlabel("t [ms]"); ax[1].set_ylabel("points"); ax[1].set_title("instants; the switches in red", fontsize=10)
G = np.load("MP7.2A_truth.npz")
lin = mp.Linear(60).grid(G["x"], G["t"])                  # the linear part at 8 W on the scoring nodes
for P_ in (5.5, 7.25, 9.5):
    gap = G[f"field_{mp.power_key(P_)}"] - P_ / mp.P0 * lin
    ax[2].plot(G["t"] * 1e3, gap.max(axis=(1, 2)), label=f"{P_} W")
    print(f"P = {P_:5.2f} W: truth - linear part, largest {gap.max():.2f} K, smallest {gap.min():.2f} K")
ax[2].axhline(2.0, color="r", lw=0.6); ax[2].set_xlabel("t [ms]"); ax[2].set_ylabel("largest of truth - linear part [K]"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** Left, the points over the die, coloured by which block is on: dense round the active block, sparse elsewhere. Middle, the instants: uniform, with peaks just after each switch. Right, what $k(T)$ adds to the exact linear part at the three test powers, at its largest over the die: it grows during each pulse, falls back as the die cools, and reaches 0.78, 1.37 and 2.36 K at 5.5, 7.25 and 9.5 W - above the 2 K asked of the field at the highest power, so the linear part alone does not meet the requirement, and its smallest value, -0.14 K at 9.5 W, shows that it is almost everywhere positive: a lower conductivity makes a hotter die.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the solver an engineer would run, on an engineering grid | the same scheme on grids fine enough to be a hundred times more exact than anything scored, extrapolated |
| grid | 20 × 20 cells of 0.5 mm, steps of 2 ms | 160 × 160 and 320 × 320 cells (0.0625 and 0.03125 mm), steps of 0.5 and 0.25 ms, combined by Richardson extrapolation |
| its job | to be **timed** against the network: the online cost of a new power | to **score** the network and the reference |

## C2 · The scheme, step by step

1. **Nodes and control volumes.** The die is covered by a square grid of nodes $h$ apart, the edges included; each interior node owns the square of side $h$ round it, its control volume. Every node of a coarse grid is a node of every finer one, and the blocks' centres and edges are nodes: no interpolation is needed to compare grids.
2. **The heat of a block.** A control volume gets the block's heat density times the share of it inside the block: 1 inside, ½ on the block's edge, ¼ on its corner. The heat put in is exact on every grid.
3. **Heat between neighbours.** The flow from one node to the next is the conductivity on the face between them - the mean of the two nodes' $k(\theta)$ - times the difference of their rises. What leaves one control volume enters the next, so heat is conserved exactly.
4. **Time: Crank–Nicolson.** Each step averages the conduction at its start and at its end: second order in time. The conductivity is taken from the field extrapolated to the middle of the step, $\tfrac32\theta^n - \tfrac12\theta^{n-1}$, which keeps second order and leaves one linear system per step.
5. **Rannacher's start.** At each switch the heat jumps, and Crank–Nicolson would carry the jump as an undamped oscillation from step to step and lose its order (the lesson of MP8.2A). The first step after each switch is taken as two backward-Euler half steps, which damp it.
6. **The linear system.** Symmetric and positive definite; solved by conjugate gradients with the diagonal as preconditioner, to a residual of $10^{-11}$.

`mp.Die(n, dt).run(P)` marches the window and returns the fields at the recorded instants, the peak, its instant and the energy balance.

In [ ]:
# The ground truth: convergence at 8 W, and the checks, from the truth file
G = np.load("MP7.2A_truth.npz")
print("cells          cell [mm]  dt [ms]   peak [K]   change of the field from the grid before [K]   energy balance off by")
for n, dt, p, c, b in zip(G["conv_cells"], G["conv_dt"], G["conv_peak"], G["conv_field_change"], G["conv_balance"]):
    print(f"{int(n):4d} x {int(n):<4d}    {mp.L/n*1e3:7.4f}   {dt*1e3:5.2f}    {p:8.4f}        {c:8.4f}                                {abs(b):.1e}")
d = np.abs(np.diff(G["conv_peak"]))
print("change of the peak between grids [K]:", np.round(d, 4), "| ratios", np.round(d[:-1] / d[1:], 1))
f = G["conv_field_change"][1:]
print("ratios of the field changes:", np.round(f[:-1] / f[1:], 1))
print(f"Richardson extrapolation: peak {float(G['peak_8W']):.4f} K; largest correction of the field {float(G['correction_8W']):.4f} K")
print(f"S = 0, no power, Ex07.2's start, against Ex07.2's exact modes: {float(G['zero_strength_modes_K']):.1e} K at most")
print(f"S = 0, the blocks on, against the exact modal sum: {float(G['zero_strength_blocks_K']):.1e} K at most")

**What you should see.** Four grids, each twice as fine in space and in time. The field at the scoring nodes changes by 0.22, 0.066 and 0.017 K, each change 3.3 and then 3.9 times smaller than the one before: the **second order** of the scheme, reached as the grid is refined. The peak changes by 0.19, 0.014 and 0.0035 K; its first change is larger because the true peak lies between the 40-cell grid's nodes, 0.25 mm apart, and its last is four times smaller than the one before. Richardson extrapolation, $(4\theta_\mathrm{fine} - \theta_\mathrm{coarse})/3$, removes the leading error; it moves the field by at most 0.0057 K, so the ground truth is within about a thousandth of a kelvin - a thousand times finer than the 1 K asked of the peak. The heat put in equals the heat through the edges plus the heat stored, to $10^{-12}$ to $10^{-10}$ of it, the tolerance of the linear solver. At $S = 0$ the same code reproduces **Ex07.2's exact two modes** to 2.8e-03 K on a 60 K start, and the exact modal sum with the blocks on to 4.5e-03 K: the code is right where the answer is known.

In [ ]:
# The energy balance at every power of the truth file
print("power [W]   heat in [J]   through the edges [J]   stored at 200 ms [J]   off by    peak [K]   at [ms]")
for P in G["powers"]:
    k = mp.power_key(P)
    if f"balance_{k}" in G:
        b = G[f"balance_{k}"]
        mark = "  <- test power" if P in mp.P_TEST else ("  <- outside the range" if P in mp.P_OUT else "")
        print(f"{P:7.2f}    {b[0]:9.4f}     {b[1]:9.4f}               {b[2]:9.4f}          {abs(b[3]/b[0]):.0e}   "
              f"{float(G[f'peak_{k}']):7.2f}   {float(G[f't_peak_{k}'])*1e3:5.0f}{mark}")

**What you should see.** The test powers and the two outside the range (4 and 11 W, for question 7). At 9.5 W the blocks put 1.71 J into the die over 180 ms; about three quarters has left through the edges by 200 ms and the rest is still stored. The balance closes to $10^{-10}$ of the heat put in. The peak rises from 16.49 K at 4 W to 47.42 K at 11 W, a little faster than the power - the nonlinearity - and always at 180 ms.

## C3 · The reference model: accuracy and time

The reference grid was chosen as the coarsest of a sequence (10, 20, 40 cells across the die, steps of 2 ms) that meets the agreed accuracy at 8 W: on 10 × 10 cells the field is 3.05 K off; on **20 × 20** the peak is 0.43 K and the field 1.32 K off; on 40 × 40, 0.21 and 0.30 K, at three times the cost. A step of 2 ms is what the 2 ms recording needs; a finer step changes nothing visible. The cell below scores the reference model against the ground truth at the test powers and times a new power. Its scoring nodes that fall between its own nodes (every other one) are interpolated linearly.

In [ ]:
# The reference model against the ground truth, at the three test powers
worst = {"peak": 0.0, "t_peak": 0.0, "field": 0.0}
for P in mp.P_TEST:
    k = mp.power_key(P)
    r = mp.reference(P)                                     # 20 x 20 cells, steps of 2 ms
    e_p, e_t = r["peak"] - float(G[f"peak_{k}"]), r["t_peak"] - float(G[f"t_peak_{k}"])
    e_f = np.abs(r["fields"] - G[f"field_{k}"]).max()
    worst = {"peak": max(worst["peak"], abs(e_p)), "t_peak": max(worst["t_peak"], abs(e_t)), "field": max(worst["field"], e_f)}
    print(f"P = {P:5.2f} W: peak {r['peak']:.2f} K (truth {float(G[f'peak_{k}']):.2f}), off by {e_p:+.2f} K, at {r['t_peak']*1e3:.0f} ms; "
          f"field off by {e_f:.2f} K at most")
t_ref = middle(lambda: mp.reference(7.0))                  # the online cost of a new power
print(f"\nreference model, 20 x 20 cells: worst peak {worst['peak']:.2f} K, worst instant {worst['t_peak']*1e3:.0f} ms, "
      f"worst field {worst['field']:.2f} K; {t_ref*1e3:.0f} ms per new power")
print(f"the network must answer a new power in under {t_ref*1e3/10:.1f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "worst_peak_K": worst["peak"], "worst_t_peak_s": worst["t_peak"], "worst_field_K": worst["field"]}   # kept for part E

**What you should see.** At the three test powers the reference model's peak is 0.28 to 0.52 K low - its nodes are 0.5 mm apart and the true peak lies between them - at the right instant, 180 ms, and its field is within 0.88 to 1.60 K, inside the agreed 1 K and 2 K; the error grows with the power. A new power takes about a tenth of a second on a CPU (64 to 160 ms in our runs on one thread, depending on what else the machine ran): a hundred steps of 2 ms, each a few dozen iterations of conjugate gradients on 361 unknowns. The time is kept in `REF`; part E compares the network against it on this same runtime.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3).

**The contract.** Your network is a PyTorch module called `net`, used as `net(x, y, t, P)`:

| argument | meaning | shape |
|---|---|---|
| `x`, `y` | the point of the die, in metres, 0 to 10 mm | `(n, 1)` |
| `t` | the instant, in seconds, 0 to 0.2 | `(n, 1)` |
| `P` | the power of each block, in watts, 5 to 10 | `(n, 1)` |
| returns | the rise $\theta$ above the package, in kelvin | `(n, 1)` |

Whatever you compute inside - features, built-in conditions, the linear part of part B, sub-networks - stays inside `net`.

## D2 · The loss, given

`losses(net, seed)` draws fresh points with part B's samplers and returns the three terms of part B's table: the heat equation with silicon's $k(T)$, its residual divided by $q_\mathrm{block}(P)$; the edges and the start, each divided by a scale of the rise, $\Theta$ = 30 K. A condition you build into `net` gives a zero term. If you build in the linear part, you will want the residual of $\delta$ instead (part B, section 2): write it in TODO 2, from `mp.Linear(M).at`, which gives $\theta_\mathrm{lin}$'s slopes and Laplacian.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

THETA = 30.0                                               # K, a scale of the rise

def losses(net, seed, n=2000):
    out = {}
    # inside the die: ρc ∂θ/∂t - div(k(θ) grad θ) - q = 0, divided by the block's heat density
    x, y, t, P = mp.sample_interior(n, seed)
    X, Y, tt = T(x).requires_grad_(True), T(y).requires_grad_(True), T(t).requires_grad_(True)
    th = net(X, Y, tt, T(P))
    k = mp.k_si(th)                                        # silicon's conductivity at the network's own rise
    flux_x, flux_y = k * d(th, X), k * d(th, Y)            # k ∂θ/∂x and k ∂θ/∂y [W/m²]
    r = mp.RHO_C * d(th, tt) - (d(flux_x, X) + d(flux_y, Y)) - T(mp.heat(x, y, t, P))
    out["pde"] = ((r / T(mp.q_block(P))) ** 2).mean()
    # the four edges: θ = 0
    rng = np.random.default_rng(seed + 1)
    s, side = rng.uniform(0, mp.L, n // 4), rng.integers(0, 4, n // 4)
    xe = np.where(side == 0, 0.0, np.where(side == 1, mp.L, s)); ye = np.where(side == 2, 0.0, np.where(side == 3, mp.L, s))
    te, Pe = mp.sample_times(n // 4, rng), rng.uniform(mp.P_MIN, mp.P_MAX, n // 4)
    out["edges"] = ((net(T(xe), T(ye), T(te), T(Pe)) / THETA) ** 2).mean()
    # the start: θ = 0 at t = 0
    xs, ys, Ps = rng.uniform(0, mp.L, n // 4), rng.uniform(0, mp.L, n // 4), rng.uniform(mp.P_MIN, mp.P_MAX, n // 4)
    out["start"] = ((net(T(xs), T(ys), T(np.zeros(n // 4)), T(Ps)) / THETA) ** 2).mean()
    return out

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`, honouring the contract above. Decide:

* **what is built in.** The edges, the start, the linear part of part B (exactly, by `mp.Linear`), the growth of $\theta$ with $P$;
* **what the network sees.** $x$, $y$ and $t$ scaled to order one; the power, scaled; features that carry the switching, if the network must follow it;
* **its structure and size.** One network of four inputs, or a sum of products of a function of space and a function of time and power; layers, neurons, activation (tanh: the second derivatives must exist).

Part E times one new power: the field at the 1 681 scoring nodes at the 101 recorded instants, 170 000 values, and the peak. A network that is evaluated 170 000 times will not be ten times faster than the reference model on a CPU; on a GPU it may be. Think about what does not change from one power to the next.

In [ ]:
# TODO 1 - your network. net(x, y, t, P) -> θ [K]; x, y in metres, t in seconds, P in watts, each (n, 1)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, x, y, t, P):
        ...                                              # return θ, shape (n, 1)

torch.manual_seed(72)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses`, or with a loss of your own built from part B (the residual of $\delta$ if you build in the linear part). Decide the optimiser or optimisers, the steps, the learning rate and its schedule, the weight of each term, the points and whether they are drawn afresh. Keep the history of the loss for TODO 3, and **measure the offline time**, any precomputation included: set `TRAIN_SECONDS`.

`quick_score()` below gives, over the three test powers, the worst peak error [K], the worst error of its instant [ms] and the worst field error [K], from the ground truth; `answer(P)` is how part E asks your network: the field at the scoring nodes at every recorded instant, and the peak, refined on a 0.025 mm patch round the hottest node.

In [ ]:
# A quick score, given: the worst errors at the test powers
G = np.load("MP7.2A_truth.npz")
XS = mp.score_nodes()                                     # the 41 scoring nodes along each edge [m]
TIMES = G["t"]                                            # the 101 recorded instants [s]
NT, NX = len(TIMES), len(XS)
_xx, _yy = np.meshgrid(XS, XS)
X_ALL, Y_ALL = T(np.tile(_xx.ravel(), NT)), T(np.tile(_yy.ravel(), NT))   # every scoring node at every instant
T_ALL = T(np.repeat(TIMES, NX * NX))

def answer(P):
    """The network's field at the scoring nodes at every recorded instant (101, 41, 41) [K], its peak [K] and
    the instant of the peak [s], for one power P [W]."""
    with torch.no_grad():
        f = net(X_ALL, Y_ALL, T_ALL, torch.full_like(X_ALL, P)).reshape(NT, NX, NX).cpu().numpy()
        i, j, k = np.unravel_index(f.argmax(), f.shape)
        px = np.clip(XS[k] + np.linspace(-0.25e-3, 0.25e-3, 21), 0, mp.L)   # a 0.025 mm patch round the hottest node
        py = np.clip(XS[j] + np.linspace(-0.25e-3, 0.25e-3, 21), 0, mp.L)
        PX, PY = np.meshgrid(px, py)
        patch = net(T(PX), T(PY), T(np.full(PX.size, TIMES[i])), T(np.full(PX.size, P)))
    return f, max(float(f.max()), float(patch.max())), float(TIMES[i])

def quick_score():
    worst = [0.0, 0.0, 0.0]
    for P in mp.P_TEST:
        k = mp.power_key(P)
        f, peak, tp = answer(P)
        worst = [max(worst[0], abs(peak - float(G[f"peak_{k}"]))), max(worst[1], abs(tp - float(G[f"t_peak_{k}"])) * 1e3),
                 max(worst[2], float(np.abs(f - G[f"field_{k}"]).max()))]
    return tuple(worst)

In [ ]:
# TODO 2 - your training
history = {}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append the loss to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst errors: peak %.2f K, its instant %.0f ms, field %.2f K" % quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss during training, and the field at 9.5 W at the instant of the peak next to the ground truth, with its error. Save each figure you want in the report as `MP7.2A_report_<name>.png` (in `cc.output_path`): part E's PDF embeds them.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the field at the scoring nodes, the peak and its instant at the three test powers, at 8 W, and outside the range (4 and 11 W);
* a sweep of the peak over 5 to 10 W;
* the **online time** of a new power on this runtime - `answer(P)`: the field at the 1 681 scoring nodes at the 101 recorded instants, and the peak - for 1, 10 and 100 new powers, against the reference model run 1, 10 and 100 times on the same runtime.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
out = {"powers": np.array(G["powers"])}
for P in G["powers"]:
    f, peak, tp = answer(float(P))
    k = mp.power_key(P)
    out[f"field_{k}"], out[f"peak_{k}"], out[f"t_peak_{k}"] = f, np.array(peak), np.array(tp)
sweep = np.linspace(mp.P_MIN, mp.P_MAX, 11)
out["sweep_P"], out["sweep_peak"] = sweep, np.array([answer(float(P))[1] for P in sweep])

# online time: a new power, the network against the reference model, on this runtime
timing = {}
for n in (1, 10, 100):
    Ps = np.linspace(5.2, 9.8, n)
    timing[f"net_{n}"] = middle(lambda: [answer(float(P)) for P in Ps])
    timing[f"ref_{n}"] = middle(lambda: [mp.reference(float(P)) for P in Ps], reps=3 if n == 100 else 5)
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP7.2A_net.pt"))   # the trained weights, beside the report
print(f"online, one new power: network {timing['net_1']*1e3:.2f} ms, reference {timing['ref_1']*1e3:.1f} ms "
      f"({timing['ref_1']/timing['net_1']:.0f} times)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's accuracy and time `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eight questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP7.2A_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): worst peak {REF['worst_peak_K']:.2f} K, worst field {REF['worst_field_K']:.2f} K, "
      f"{REF['t_ref_s']*1e3:.0f} ms per new power")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* at each test power: the peak error [K], the error of its instant [ms] and the largest field error [K];
* the network's online time for one new power, the reference model's on the same runtime (both in `R`), and their ratio;
* the offline training time.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>45}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP7.2A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the largest field error against time at the three test powers; the peak against the power from 4 to 11 W, the network's and the ground truth's, with the linear model's for comparison; and the online times of the network and the reference model for 1, 10 and 100 powers (`R["net_1"]`, `R["ref_10"]` and so on).

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eight questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 4 are about your network and its result; 5 to 7 ask you to use it; 8 concludes. The next cell writes `MP7.2A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: what it sees, what is built into its form (the edges, the start, the linear part, the power), its structure, layers and neurons, and its number of weights. Why this design?": """
""",
    "2. Your training: the loss terms and their scaling, the points, the optimiser(s), steps and learning rates, and the offline time in seconds, any precomputation included. What was hardest to bring down?": """
""",
    "3. The peak and its instant at 5.5, 7.25 and 9.5 W: give each error against the ground truth. Are they within 1 K and 2 ms, and how do they compare with the reference model's?": """
""",
    "4. The field at the test powers: your largest error, where in the die and at which instant it occurs, and why there. Show it in a figure.": """
""",
    "5. The online time for a new power: your network against the reference model on the same runtime, in milliseconds, and the ratio; then for 10 and 100 powers. What does each spend its time on?": """
""",
    "6. How much does silicon's k(T) add to the peak at the three test powers (ground truth against the linear model of part A), and how much does your network give? Is a linear model enough for this die?": """
""",
    "7. Outside the training range: your peak and field errors at 4 W and 11 W. Do they grow outside the range, and why, or why not, for your design?": """
""",
    "8. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? What would it take to answer another schedule of the blocks?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP7.2A — A Die with a Moving Hot Spot", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP7.2A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP7.2A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP7.2A_report.md into MP7.2A_report.pdf, with any figure
# saved as MP7.2A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP7.2A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP7.2A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP7.2A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP7.2A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass